In [1]:
from datasets import load_dataset
from collections import Counter
import gc
import ast
import pandas as pd

In [4]:
ignored_tags = open("data/ignored_tags.txt", "r").read().splitlines()
print(ignored_tags[-10:])

['historical_name_connection', 'color_connection', 'hair_color_connection', 'country_connection', 'character_age', 'content_rating', 'style_parody', 'real_life', 'borrowed_character', 'look-alike']


In [7]:
def clean(data_path, new_path, itags):
    df = pd.read_parquet(data_path)
    print("len before:", len(df))
    df = df[
        df["tags"].apply(
            lambda tags: any(tag not in itags for tag in tags)
        )
    ].copy()
    # clean other tags
    df["tags"] = df["tags"].apply(
        lambda tags: [tag for tag in tags if tag not in itags]
    )
    df = df[df["tags"].str.len() > 0].reset_index(drop=True)
    print("len after:", len(df))
    df.to_parquet(
        new_path,
        index=False,
        compression="zstd",
    )

In [8]:
clean("data/danbooru2025_val.parquet", "data/danbooru2025_val.parquet", ignored_tags)

len before: 100000
len after: 100000


In [9]:
clean("data/danbooru2025_train.parquet", "data/danbooru2025_train.parquet", ignored_tags)

len before: 1009130
len after: 1009130


In [10]:
# validation
def get_tags(data_path):
    df = pd.read_parquet(data_path)
    ftags = Counter()
    for tags in df["tags"]:
        ftags.update(tags)
    return ftags

In [13]:
ftags = get_tags("data/danbooru2025_val.parquet")
"real_life" in ftags

False